Imports & Init

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd

build_dir = "/content/drive/MyDrive/NLP/Project/"
output_dir_dataset = os.path.join(build_dir, "Dataset/BigOBenchComplexityMerged/")

train_data = pd.read_csv(os.path.join(output_dir_dataset, "train_dataset.csv"))
val_data = pd.read_csv(os.path.join(output_dir_dataset, "val_dataset.csv"))

# Preprocess & Data

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-small")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/703k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/294k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/12.5k [00:00<?, ?B/s]

In [ ]:
def preprocess(example, tokenizer, max_input_len=512, max_output_len=256):
    input_text = "solve: " + example["description"]
    target_text = example["solution_code"]

    inputs = tokenizer(input_text, max_length=max_input_len, truncation=True, padding="max_length")
    targets = tokenizer(target_text, max_length=max_output_len, truncation=True, padding="max_length")

    return {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "labels": targets["input_ids"]
    }

In [ ]:
from torch.utils.data import Dataset

class BigOBenchDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data  # Store the raw data instead of preprocessed data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example = self.data[idx]  # Get the raw data for the current index
        x = preprocess(example, self.tokenizer)  # Preprocess on-the-fly
        return {
            "input_ids": torch.tensor(x["input_ids"]),
            "attention_mask": torch.tensor(x["attention_mask"]),
            "labels": torch.tensor(x["labels"])
        }

In [ ]:
train_data_dict = train_data[["description", "solution_code"]].to_dict(orient="records")
val_data_dict = val_data[["description", "solution_code"]].to_dict(orient="records")

train_dataset = BigOBenchDataset(train_data_dict, tokenizer)
val_dataset = BigOBenchDataset(val_data_dict, tokenizer)

# Model(Pretrained CodeT5 small)

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("Salesforce/codet5-small")

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

# Fine Tuning

In [9]:
!pip install codebleu

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    # Note: CodeBLEU expects lists of strings
    return {
        "codebleu": calc_code_bleu(decoded_labels, decoded_preds, lang="python")["codebleu"]
    }

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./codet5-small-finetuned",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_dir="./logs",
    logging_steps=50,
    fp16=True,
    report_to="none"
)

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

<ipython-input-14-6cf689081aa5>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
import torch

trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/data/data_collator.py:741: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
50,2.301600
100,1.291900
150,1.176600
200,1.105300
250,1.082400
300,1.056600
350,0.971800
400,0.900400
450,1.019300


KeyboardInterrupt: 

# Inference

In [ ]:
problem_inference = "solve: Write a Python function to return the square of a number."

In [ ]:
inputs = tokenizer(problem_inference, return_tensors="pt", padding=True).to(model.device)

outputs = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=256, # To adjust
    num_beams=5,    # beam search
    early_stopping=True
)

generated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_code)

a,b,c=map(int,input().split()) return a[0]%2==0.0 and a[1]%2==0 and a[2]%2==0: return a[0]%2==0: return a[0]%2==0: return a[0]%2==0 and a[2]%2==0 and a[0]%2==0 and a[1]%2==0: return a[0]%2==0 and a[0]%2==0 and a[1]%2==0 and a[0]%2==0 and a[1]%2==0 and a[1]%2==0 and a[1]%2==0 and a[1]%2==0 and a[1]%2==0 and a[1]%2==0 and a[1]%2==0 and a[0]%2==0 and a[1]%2==0 and a[0]%2==0 and a[1]%2==0 and a[1]%2==0 and a[2]%2==0


# Evaluation

## Evaluation with CodeBleu

In [1]:
!pip install git+https://github.com/k4black/codebleu.git
!pip install tree-sitter-python

  Cloning https://github.com/k4black/codebleu.git to /tmp/pip-req-build-1u6upq5j
  Running command git clone --filter=blob:none --quiet https://github.com/k4black/codebleu.git /tmp/pip-req-build-1u6upq5j
  Resolved https://github.com/k4black/codebleu.git to commit b0edb622f6a52fe9d1edc407be5061d3e1462a7f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


Example usage of [codebleu](https://github.com/k4black/codebleu/tree/main)

In [4]:
from codebleu import calc_codebleu

# Example Code
generated_code = """def sum_even_numbers(nums):
    return sum(filter(lambda x: x % 2 == 0, nums))"""

reference_code = """def sum_even_numbers(nums):
    return sum(num for num in nums if num % 2 == 0)"""

# Calculate CodeBLEU
score = calc_codebleu([reference_code], [generated_code], "python")
print("CodeBLEU Score Breakdown:", score)


CodeBLEU Score Breakdown: {'codebleu': 0.2774712076731169, 'ngram_match_score': 0.11944970027158508, 'weighted_ngram_match_score': 0.11770785769360982, 'syntax_match_score': 0.2727272727272727, 'dataflow_match_score': 0.6}


Function that given the test set, returns the average codebleu metrics, in order to evaluate on the test set

In [7]:
from codebleu import calc_codebleu

def evaluate_test_set(test_set):
    """
    Evaluate a model on a test set using CodeBLEU.

    Args:
    - test_set (list of dict): Each dict contains:
        - 'problem': Problem description.
        - 'references': List of reference solutions (strings).
        - 'generated': Generated solution (string).

    Returns:
    - dict: Average scores across the entire test set.
    """
    total_scores = {
        "codebleu": 0.0,
        "ngram_match_score": 0.0,
        "weighted_ngram_match_score": 0.0,
        "syntax_match_score": 0.0,
        "dataflow_match_score": 0.0
    }

    num_problems = len(test_set)

    for problem_data in test_set:
        references = problem_data["references"]
        generated = problem_data["generated"]

        # Calculate CodeBLEU scores for each reference
        scores = [calc_codebleu([ref], [generated], "python") for ref in references]

        # Between all possible solutions, taking the one with the best codebleu
        best_score = max(scores, key=lambda x: x['codebleu'])

        for key in total_scores.keys():
            total_scores[key] += best_score[key]

    # Calculate average scores
    avg_scores = {key: value / num_problems for key, value in total_scores.items()}

    return avg_scores

In [8]:
# Example Test Set
test_set = [
    {
        "problem": "Sum of even numbers in a list",
        "references": [
            """def sum_even_numbers(nums):
    return sum(num for num in nums if num % 2 == 0)""",
            """def sum_even_numbers(arr):
    return sum(x for x in arr if x % 2 == 0)"""
        ],
        "generated": """def sum_even_numbers(nums):
    return sum(filter(lambda x: x % 2 == 0, nums))"""
    },
    {
        "problem": "Find the maximum number in a list",
        "references": [
            """def find_max(arr):
    return max(arr)""",
            """def find_max(arr):
    max_val = float('-inf')
    for num in arr:
        if num > max_val:
            max_val = num
    return max_val"""
        ],
        "generated": """def find_max(nums):
    return max(nums)"""
    }
]

results = evaluate_test_set(test_set)
print("Average CodeBLEU Scores Across Test Set:")
for key, value in results.items():
    print(f"{key}: {value:.4f}")


Average CodeBLEU Scores Across Test Set:
codebleu: 0.4337
ngram_match_score: 0.1466
weighted_ngram_match_score: 0.1517
syntax_match_score: 0.6364
dataflow_match_score: 0.8000


# Evaluation with unit tests

Each problem has a _"tests"_ field, that is a dict with public_tests, private_tests and generated_tests. For the evaluation we are interested mostly in public and generated tests, since the private ones could contains errors that would mislead the results.

Function to execute the code

In [1]:
import traceback

def execute_code(code_str, func_name, inputs):
    """
    Execute the provided code and call the specified function with given inputs.

    Args:
    - code_str (str): The complete function code as a string.
    - func_name (str): The name of the function to call.
    - inputs (tuple): The input arguments for the function.

    Returns:
    - dict: Contains either 'result' or 'error'.
    """
    try:
        # Prepare the execution environment
        exec_globals = {}
        exec(code_str, exec_globals)

        # Access the function
        func = exec_globals.get(func_name)
        if not func:
            return {"error": f"Function '{func_name}' not found."}

        # Execute the function with inputs
        if not isinstance(inputs, tuple):
            inputs = (inputs,)

        result = func(*inputs)
        return {"result": result}

    except Exception as e:
        return {"error": traceback.format_exc()}

Function, that given the generated code, and the test cases to execute, executes them and return the results

In [2]:
def run_code(code_str, func_name, test_cases):
    """
    Run the function with multiple test cases and generate a test report.

    Args:
    - code_str (str): The complete function code as a string.
    - func_name (str): The name of the function to call.
    - test_cases (list of dict): Each dict contains:
        - 'input': Input arguments.
        - 'expected_output': Expected output.

    Returns:
    - dict: Test report.
    """
    report = {
        "total": len(test_cases),
        "passed": 0,
        "failed": 0,
        "errors": []
    }

    for i, test in enumerate(test_cases):
        inputs = test["input"]
        expected = test["expected_output"]

        # Execute the function
        result_data = execute_code(code_str, func_name, inputs)

        # Check for errors
        if "error" in result_data:
            report["failed"] += 1
            report["errors"].append({
                "test_case": i + 1,
                "input": inputs,
                "expected": expected,
                "error": result_data["error"]
            })
        else:
            # Check the output
            result = result_data["result"]
            if result == expected:
                report["passed"] += 1
            else:
                report["failed"] += 1
                report["errors"].append({
                    "test_case": i + 1,
                    "input": inputs,
                    "expected": expected,
                    "received": result
                })

    return report

Example usage of the above function

In [3]:
# Example Code
generated_code = """
def sum_even_numbers(nums):
    return sum(filter(lambda x: x % 2 == 0, nums))
"""

# Test Cases for "sum_even_numbers"
test_cases = [
    {"input": [1, 2, 3, 4], "expected_output": 6},
    {"input": [1, 3, 5], "expected_output": 0},
    {"input": [2, 4, 6], "expected_output": 12},
    {"input": [], "expected_output": 0},
    {"input": [0], "expected_output": 0}
]

# Run Evaluation
result = run_code(generated_code, "sum_even_numbers", test_cases)
print("Test Report:", result)

Test Report: {'total': 5, 'passed': 5, 'failed': 0, 'errors': []}


**ISSUE**: the solution can be correct, but it can be written without a specific function.

**POSSIBLE SOLUTION**: for LLMs we could enforce in the prompt to return always a standardized way of writing that. We can try to have that with smaller models.

However, generated codes does not look like _generated_code_. Hence, we need another function that handles this, preparing the code to be given to _run_code()_.

In [5]:
def prepare_code(solution_code):
    """
    Convert a single-line string with \n characters into a properly formatted multiline string.

    Args:
    - solution_code (str): The solution code with single \n characters for newlines.

    Returns:
    - str: The formatted code ready for use in `run_code()`.
    """
    # The code already has single `\n` characters representing actual newlines.
    # We only need to encapsulate it in triple quotes for `exec()` compatibility.
    formatted_code = f'"""\n{solution_code}\n"""'
    return formatted_code.strip()

# Example Usage
raw_solution_code = "s = input()\nm = int(input())\nk1 = k2 = 0\nz = [0]*1001; index = 0\nx = True\np = list(s)\nif p.count('1') == 3:\n    i = 0\n    while p[i] == '0':\n        i += 1\n    i += 1\n    while p[i] == '0':\n        i += 1\n    z[1] = i+1\n    k2 = z[1]\n    p = 1\n    index = 1\nelse: p = 0\nfor j in range(p, m):\n    for i in range(10):\n        if s[i] != '0' and i+1 != z[index] and k1 + (i+1) > k2:\n            k1 += (i+1)\n            index += 1\n            z[index] = i+1;\n            break\n    else:\n        x = False\n        break\n    if index == m:\n        break\n    for i in range(10):\n        if s[i] != '0' and i+1 != z[index] and k2 + (i+1) > k1:\n            k2 += (i+1)\n            index += 1\n            z[index] = i+1\n            break\n    else:\n        x = False\n        break\n    if index == m:\n        break\nif x:\n    print('YES')\n    for i in range(1, m+1):\n        print(z[i], end=' ')\nelse:\n    print('NO')\n"

# Prepare the code
formatted_code = prepare_code(raw_solution_code)

# Verify the output
print("Formatted Code:\n", formatted_code)

Formatted Code:
 """
s = input()
m = int(input())
k1 = k2 = 0
z = [0]*1001; index = 0
x = True
p = list(s)
if p.count('1') == 3:
    i = 0
    while p[i] == '0':
        i += 1
    i += 1
    while p[i] == '0':
        i += 1
    z[1] = i+1
    k2 = z[1]
    p = 1
    index = 1
else: p = 0
for j in range(p, m):
    for i in range(10):
        if s[i] != '0' and i+1 != z[index] and k1 + (i+1) > k2:
            k1 += (i+1)
            index += 1
            z[index] = i+1;
            break
    else:
        x = False
        break
    if index == m:
        break
    for i in range(10):
        if s[i] != '0' and i+1 != z[index] and k2 + (i+1) > k1:
            k2 += (i+1)
            index += 1
            z[index] = i+1
            break
    else:
        x = False
        break
    if index == m:
        break
if x:
    print('YES')
    for i in range(1, m+1):
        print(z[i], end=' ')
else:
    print('NO')

"""


So now we have just to mix prepare_code() and run_code()

In [7]:
# Test Example
raw_solution_code = "def sum_even_numbers(nums):\n    return sum(filter(lambda x: x % 2 == 0, nums))"

# Test Cases
test_cases = [
    {"input": [1, 2, 3, 4], "expected_output": 6},
    {"input": [1, 3, 5], "expected_output": 0},
    {"input": [2, 4, 6], "expected_output": 12}
]

# Prepare the code
generated_code = prepare_code(raw_solution_code)

# Run the evaluation
result = run_code(generated_code, "sum_even_numbers", test_cases)
print("Test Report:", result)


Test Report: {'total': 3, 'passed': 0, 'failed': 3, 'errors': [{'test_case': 1, 'input': [1, 2, 3, 4], 'expected': 6, 'error': "Function 'sum_even_numbers' not found."}, {'test_case': 2, 'input': [1, 3, 5], 'expected': 0, 'error': "Function 'sum_even_numbers' not found."}, {'test_case': 3, 'input': [2, 4, 6], 'expected': 12, 'error': "Function 'sum_even_numbers' not found."}]}


# Evaluation with problem categorization

**TODO:** organize the test set problems by type(arrays,strings,BFS, DFS, ...) and by difficulty levels(see codeforces metadata).

This way we can say per model how well he is behaving on different type of problems and on different difficulty levels.

When doing so we'd like to mix up codebleu and unit tests evaluations